<a href="https://colab.research.google.com/github/nikolas-joyce/macro-signal-strategies/blob/main/01-excess-inflation/notebooks/excess_inflation_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Excess Inflation Strategy

> Measures how far realised inflation deviates from the Fed's 2% target and uses that deviation to time equity and bond exposure. High excess inflation → rotate defensive (TLT, cash). Low or negative excess inflation → risk-on (SPY). Signal is constructed from FRED CPI data, z-scored over a 5-year rolling window, and lagged one month to respect data availability.

## Signal logic
- **Raw signal**: 6-month annualised CPI change − 2% target, scaled by target
- **Normalisation**: rolling 60-month z-score, clipped ±3σ
- **Lag**: 1 month (CPI released ~3 weeks after reference month — trade at next month open)
- **Variants**: headline 6m, core 6m, headline 3m

## Strategy variants
| Variant | Rule |
|---------|------|
| Directional | Long SPY when signal < −1, long TLT when signal > +1, cash otherwise |
| Long-only overlay | Long SPY when signal < 0, cash when signal ≥ 0 |
| Continuous | Position = −clip(signal, −2, 2) / 2 (scaled ±100%) |
| Relative value | Long SPY / short TLT when signal < −1; reverse when signal > +1 |

## Notebook structure
| Section | Description |
|---------|-------------|
| 0 | Install & imports |
| 1 | Config & API keys |
| 2 | Data — FRED CPI + yfinance ETFs |
| 3 | Signal construction & variants |
| 4 | Signal validation & predictive power |
| 5 | Backtest — all four variants |
| 6 | Performance comparison |
| 7 | Regime analysis |
| 8 | Episode deep-dives |
| 9 | Parameter sensitivity |

**Run all cells top-to-bottom in a fresh Colab runtime.**

> **API key required**: Add `FRED_API_KEY` to Colab Secrets (key icon, left sidebar).
> Free key at https://fred.stlouisfed.org/docs/api/api_key.html — takes 30 seconds.


## Section 0 — Install & Imports

In [ ]:
!pip install fredapi yfinance --quiet
print("Dependencies installed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

try:
    from fredapi import Fred
    import yfinance as yf
except ImportError as e:
    raise ImportError(f'Missing dependency: {e}. Restart runtime after install.') from e

print('Imports complete')


## Section 1 — Config & API Keys

In [ ]:
from google.colab import userdata
FRED_API_KEY = userdata.get('FRED_API_KEY')  # set via Colab Secrets

START_DATE = '1990-01-01'
COST_BPS   = 5

# ETF universe
TICKERS = {
    'SPY': 'US Equity',
    'TLT': 'Long Treasury',
    'GLD': 'Gold',
    'UUP': 'USD',
}

# Signal parameters
Z_WINDOW   = 60    # months for rolling z-score
CLIP_SIGMA = 3.0   # clip signal at ±3σ

print(f'Config set. Start: {START_DATE}')


## Section 2 — Data

In [ ]:
fred = Fred(api_key=FRED_API_KEY)

print('Pulling CPI series from FRED...')
cpi_headline = fred.get_series('CPIAUCSL', observation_start=START_DATE).resample('MS').last()
cpi_core     = fred.get_series('CPILFESL', observation_start=START_DATE).resample('MS').last()

print(f'  Headline CPI : {cpi_headline.index[0].date()} → {cpi_headline.index[-1].date()} ({len(cpi_headline)} obs)')
print(f'  Core CPI     : {cpi_core.index[0].date()} → {cpi_core.index[-1].date()} ({len(cpi_core)} obs)')

print('\nDownloading ETF prices...')
raw     = yf.download(list(TICKERS.keys()), start=START_DATE, auto_adjust=True, progress=False)
prices  = raw['Close'].resample('MS').first()   # month-start price (trade at open)
returns = prices.pct_change()

# Align monthly index
common = cpi_headline.index.intersection(prices.index)
cpi_headline = cpi_headline.reindex(common)
cpi_core     = cpi_core.reindex(common)
prices       = prices.reindex(common)
returns      = returns.reindex(common)

print(f'  ETF shape    : {prices.shape}')
print(f'  Common range : {common[0].date()} → {common[-1].date()}')
print('Data ready')


## Section 3 — Signal Functions

In [ ]:
def annualise_pct(series, periods):
    """Convert n-period % change to annualised: (1+r)^(12/n) - 1."""
    raw = series.pct_change(periods)
    return raw.apply(lambda x: (1 + x) ** (12 / periods) - 1) * 100


def excess_inflation(cpi, target_pct=2.0, periods=6):
    """Raw excess inflation signal: annualised CPI change minus target, scaled."""
    ann = annualise_pct(cpi, periods)
    return (ann - target_pct) / target_pct


def normalise_signal(raw, window=Z_WINDOW, clip=CLIP_SIGMA):
    """Rolling z-score, clipped, lagged 1 month."""
    mu    = raw.rolling(window, min_periods=window // 2).mean()
    sigma = raw.rolling(window, min_periods=window // 2).std().replace(0, np.nan)
    z     = (raw - mu) / sigma
    return z.clip(-clip, clip).shift(1)   # lag-1: trade next month open


print('Signal functions defined')


## Section 4 — Signal Construction & Variants

In [ ]:
# Primary signal: headline CPI, 6-month annualised, z-scored
raw_headline_6m = excess_inflation(cpi_headline, periods=6)
signal          = normalise_signal(raw_headline_6m)

# Variants
variants = pd.DataFrame({
    'headline_6m': signal,
    'core_6m':     normalise_signal(excess_inflation(cpi_core,     periods=6)),
    'headline_3m': normalise_signal(excess_inflation(cpi_headline, periods=3)),
})

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

signal.plot(ax=axes[0], color='steelblue', label='Headline 6m (primary)')
axes[0].axhline(0,  color='black',  linewidth=0.8, alpha=0.5)
axes[0].axhline(1,  color='red',    linewidth=0.8, linestyle='--', alpha=0.7, label='+1σ threshold')
axes[0].axhline(-1, color='green',  linewidth=0.8, linestyle='--', alpha=0.7, label='−1σ threshold')
axes[0].set_title('Excess Inflation Signal (z-score, 60m window, lag-1)')
axes[0].legend()

variants.plot(ax=axes[1], title='Signal Variants — Headline vs Core, 6m vs 3m')
axes[1].axhline(0, color='black', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f'Signal range: {signal.dropna().index[0].date()} → {signal.dropna().index[-1].date()}')
print(f'Current reading: {signal.iloc[-1]:.2f}σ  ({"risk-off" if signal.iloc[-1] > 0 else "risk-on"})')


## Section 5 — Signal Validation

In [ ]:
valid = signal.dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Time series
valid.plot(ax=axes[0], title='Signal over time')
axes[0].axhline(0, color='black', linewidth=0.8)

# Distribution
axes[1].hist(valid, bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Signal distribution')
axes[1].axvline(0, color='black', linewidth=0.8)

# Autocorrelation
lags  = range(1, 13)
acf   = [valid.autocorr(lag) for lag in lags]
axes[2].bar(lags, acf, color='steelblue')
axes[2].set_title('Autocorrelation (months)')
axes[2].set_xlabel('Lag')
axes[2].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print(f'Signal stats:')
print(f'  Mean : {valid.mean():.3f}')
print(f'  Std  : {valid.std():.3f}')
print(f'  % positive : {(valid > 0).mean():.1%}')
print(f'  % > +1σ    : {(valid > 1).mean():.1%}')
print(f'  % < -1σ    : {(valid < -1).mean():.1%}')


## Section 6 — Predictive Power

In [ ]:
spy_fwd = returns['SPY'].shift(-1)   # next-month SPY return
aligned = pd.concat([signal, spy_fwd], axis=1).dropna()
aligned.columns = ['signal', 'fwd_ret']

# Rolling IC
ic = aligned['signal'].rolling(24).corr(aligned['fwd_ret'])

# Quintile analysis
aligned['quintile'] = pd.qcut(aligned['signal'], 5, labels=False, duplicates='drop')
q_means = aligned.groupby('quintile')['fwd_ret'].mean() * 12   # annualised

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ic.plot(ax=axes[0], color='steelblue', title='Rolling 24m Information Coefficient (signal vs fwd SPY return)')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('IC (Pearson)')

colors = ['green' if v < 0 else 'red' for v in q_means]   # low inflation = positive
q_means.plot(kind='bar', ax=axes[1], color=colors,
             title='Signal Quintile vs Mean Annualised Fwd SPY Return')
axes[1].set_xlabel('Quintile (1 = low inflation, 5 = high)')
axes[1].set_ylabel('Ann. Return')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

mean_ic = ic.dropna().mean()
print(f'Mean IC: {mean_ic:.3f}  ({"positive predictive power" if mean_ic < 0 else "inverse — high inflation = bad for SPY"})')


## Section 7 — Backtest Functions

In [ ]:
def apply_costs(ret_series, position, cost_bps=COST_BPS):
    cost    = cost_bps / 10_000
    pos     = position.reindex(ret_series.index).fillna(0)
    turnover = pos.diff().abs().fillna(0)
    return ret_series * pos - turnover * cost


def run_backtest(sig, returns, variant='directional'):
    """
    sig     : monthly signal series (z-score)
    returns : DataFrame of monthly ETF returns
    variant : directional | overlay | continuous | relvalue
    """
    spy = returns['SPY']
    tlt = returns['TLT'] if 'TLT' in returns.columns else spy * 0

    if variant == 'directional':
        pos_spy = pd.Series(np.where(sig < -1, 1, np.where(sig > 1, 0, 0)), index=sig.index)
        pos_tlt = pd.Series(np.where(sig >  1, 1, 0), index=sig.index)
        ret = apply_costs(spy, pos_spy) + apply_costs(tlt, pos_tlt)

    elif variant == 'overlay':
        pos_spy = (sig < 0).astype(float)
        ret = apply_costs(spy, pos_spy)

    elif variant == 'continuous':
        pos_spy = (-sig.clip(-2, 2) / 2).fillna(0)
        ret = apply_costs(spy, pos_spy)

    elif variant == 'relvalue':
        pos_spy = pd.Series(np.where(sig < -1,  1, np.where(sig > 1, -1, 0)), index=sig.index)
        pos_tlt = pd.Series(np.where(sig >  1,  1, np.where(sig < -1, -1, 0)), index=sig.index)
        ret = apply_costs(spy, pos_spy) + apply_costs(tlt, pos_tlt)

    return ret.rename(variant)


def performance_summary(r, name=''):
    r = r.dropna()
    if len(r) < 12:
        return pd.Series([np.nan]*6,
            index=['Ann.Ret','Ann.Vol','Sharpe','Max DD','Calmar','Win Rate'], name=name)
    ann_ret = (1 + r.mean()) ** 12 - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-9 else np.nan
    cum     = (1 + r).cumprod()
    mdd     = (cum / cum.cummax() - 1).min()
    calmar  = ann_ret / abs(mdd) if abs(mdd) > 1e-9 else np.nan
    hits    = (r > 0).mean()
    return pd.Series({
        'Ann.Ret': ann_ret, 'Ann.Vol': ann_vol, 'Sharpe': sharpe,
        'Max DD': mdd, 'Calmar': calmar, 'Win Rate': hits,
    }, name=name)


print('Backtest functions defined')


## Section 8 — Run Backtest

In [ ]:
variants_list = ['directional', 'overlay', 'continuous', 'relvalue']
strategy_rets = {}

sig_aligned = signal.reindex(returns.index)

for v in variants_list:
    strategy_rets[v] = run_backtest(sig_aligned, returns, variant=v)

# SPY buy-and-hold benchmark
spy_bh = returns['SPY'].rename('SPY B&H')

print('Backtests complete')
for v in variants_list:
    r = strategy_rets[v].dropna()
    print(f'  {v:15s}  {len(r)} months  ({r.index[0].date()} – {r.index[-1].date()})')


## Section 9 — Performance

In [ ]:
results = pd.concat([
    performance_summary(strategy_rets[v], name=v) for v in variants_list
] + [performance_summary(spy_bh, name='SPY B&H')], axis=1).T

print('Strategy Performance Summary')
print('=' * 70)
print(results.round(3).to_string())

# Equity curves
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]})

colors = {'directional': 'steelblue', 'overlay': 'darkorange',
          'continuous': 'seagreen', 'relvalue': 'purple', 'SPY B&H': 'grey'}

for v in variants_list:
    r   = strategy_rets[v].dropna()
    cum = (1 + r).cumprod()
    cum.plot(ax=axes[0], label=v, color=colors[v])

cum_spy = (1 + spy_bh.dropna()).cumprod()
cum_spy.plot(ax=axes[0], label='SPY B&H', color='grey', linestyle='--', linewidth=1.5)

axes[0].set_title('Excess Inflation Strategy — Equity Curves (monthly rebalance)')
axes[0].set_ylabel('Cumulative return')
axes[0].legend()

# Drawdown of directional strategy
r_dir = strategy_rets['directional'].dropna()
cum_d = (1 + r_dir).cumprod()
dd    = (cum_d / cum_d.cummax() - 1)
dd.plot(ax=axes[1], color='firebrick', label='Directional drawdown')
axes[1].set_ylabel('Drawdown')
axes[1].legend()
plt.tight_layout()
plt.show()


## Section 10 — Regime Analysis

In [ ]:
# 4-quadrant regime: Growth (SPY 12m return) x Inflation (signal level)
# High growth + low inflation = best for equities
# Low growth + high inflation = stagflation = worst

spy_12m     = returns['SPY'].rolling(12).sum()
growth_hi   = spy_12m > spy_12m.rolling(60).median()
inflation_hi = signal > 0

regimes = pd.Series('Unknown', index=signal.index)
regimes[growth_hi  & ~inflation_hi] = 'Goldilocks (growth↑, infl↓)'
regimes[growth_hi  &  inflation_hi] = 'Overheating (growth↑, infl↑)'
regimes[~growth_hi & ~inflation_hi] = 'Deflation (growth↓, infl↓)'
regimes[~growth_hi &  inflation_hi] = 'Stagflation (growth↓, infl↑)'

regime_sharpe = {}
for regime in regimes.unique():
    mask = regimes == regime
    r    = strategy_rets['directional'][mask].dropna()
    if len(r) > 6:
        ann = (1 + r.mean()) ** 12 - 1
        vol = r.std() * np.sqrt(12)
        regime_sharpe[regime] = ann / vol if vol > 1e-9 else np.nan

regime_df = pd.Series(regime_sharpe).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regime_df.plot(kind='barh', ax=axes[0],
    title='Directional Strategy — Sharpe by Macro Regime',
    color=['red' if v < 0 else 'steelblue' for v in regime_df])
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Sharpe ratio')

# Regime frequency
freq = regimes.value_counts() / len(regimes)
freq.plot(kind='pie', ax=axes[1], title='Regime Frequency',
          autopct='%1.0f%%', startangle=90)

plt.tight_layout()
plt.show()

print('Regime Sharpe ratios:')
print(regime_df.round(2).to_string())


## Section 11 — Episode Deep-Dives

In [ ]:
EVENTS = {
    'GFC':             ('2007-01-01', '2009-12-01'),
    'Post-GFC (QE)':   ('2009-01-01', '2013-12-01'),
    '2022 CPI shock':  ('2021-06-01', '2023-06-01'),
    'COVID':           ('2020-01-01', '2021-06-01'),
}

fig, axes = plt.subplots(len(EVENTS), 2, figsize=(14, 4 * len(EVENTS)))

for row, (episode, (start, end)) in enumerate(EVENTS.items()):
    ep_sig = signal.loc[start:end]
    ep_ret = strategy_rets['directional'].loc[start:end]
    ep_spy = returns['SPY'].loc[start:end]

    # Signal
    ep_sig.plot(ax=axes[row, 0], color='steelblue', title=f'{episode} — Signal')
    axes[row, 0].axhline(0,  color='black', linewidth=0.8)
    axes[row, 0].axhline(1,  color='red',   linewidth=0.8, linestyle='--', alpha=0.7)
    axes[row, 0].axhline(-1, color='green', linewidth=0.8, linestyle='--', alpha=0.7)
    axes[row, 0].set_ylabel('Z-score')

    # Equity curve vs SPY
    cum_strat = (1 + ep_ret.fillna(0)).cumprod()
    cum_spy_e = (1 + ep_spy.fillna(0)).cumprod()
    cum_strat.plot(ax=axes[row, 1], label='Directional', color='steelblue',
                   title=f'{episode} — Strategy vs SPY')
    cum_spy_e.plot(ax=axes[row, 1], label='SPY B&H', color='grey', linestyle='--')
    axes[row, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Section 12 — Parameter Sensitivity

In [ ]:
import itertools

sweep_rows = []
for window, threshold in itertools.product([36, 48, 60, 72, 84], [0.5, 1.0, 1.5]):
    def _run(sig, ret, thr):
        pos_spy = pd.Series(np.where(sig < -thr, 1, np.where(sig > thr, 0, 0)), index=sig.index)
        pos_tlt = pd.Series(np.where(sig >  thr, 1, 0), index=sig.index)
        r = apply_costs(ret['SPY'], pos_spy) + apply_costs(ret['TLT'], pos_tlt)
        return r

    sig_w = normalise_signal(raw_headline_6m, window=window)
    sig_a = sig_w.reindex(returns.index)
    r     = _run(sig_a, returns, threshold)
    s     = performance_summary(r)
    sweep_rows.append({**s.to_dict(), 'z_window': window, 'threshold': threshold})

sweep_df = pd.DataFrame(sweep_rows).sort_values('Sharpe', ascending=False)
print('Parameter Sweep — Z-score window × Entry threshold (Directional variant):')
print(sweep_df[['z_window','threshold','Sharpe','Ann.Ret','Max DD','Calmar']].round(3).to_string())
